<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/13-performance-tuning/05-pre-ship-checklist.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)


def spark_ui(spark=None):
    """Open the Spark UI — and make it reachable from Colab.

    The UI is served by the driver on the *runtime*, so on Colab plain
    http://localhost:4040 is not something your browser can reach. Colab can
    proxy a runtime port, so ask it for a real URL. Call this AFTER the
    SparkSession exists, or there is no server on the port yet.
    """
    port = 4040
    if spark is not None:  # the driver bumps the port if 4040 was taken
        try:
            port = int(spark.sparkContext.uiWebUrl.rsplit(":", 1)[1])
        except Exception:
            pass
    if not IN_COLAB:
        return f"http://localhost:{port}"
    from google.colab.output import eval_js

    url = eval_js(f"google.colab.kernel.proxyPort({port})")
    print(f"Spark UI: {url}  (open in a new tab)")
    return url


print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))
print("Spark UI: call spark_ui(spark) after creating the session.")


# 13.5 — Pre-ship checklist, applied: fix a deliberately bad script

A script with FIVE checklist violations baked in, on purpose. Walk
through the checklist, catch each one via evidence (not vibes), fix
it, and re-measure. This is the module's capstone-style close.

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("13.5-pre-ship-checklist")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "false")  # stable teaching plans (Module 7/10 convention)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

## The bad script

Realistic-looking, and wrong/slow in five specific, checklist-covered
ways. Read it before running — try to spot the violations yourself
first.

In [ ]:
from pyspark.sql.types import DoubleType

# --- THE BAD SCRIPT (deliberately flawed) -----------------------------

raw_orders = spark.range(3_000_000).withColumn(
    "customer_id", F.when(col("id") % 100 < 85, F.lit(1))       # VIOLATION 1: hidden skew
                     .otherwise((col("id") % 100).cast("int")))
raw_orders = raw_orders.withColumn("amount", (col("id") % 500).cast("double"))

@F.udf(DoubleType())
def apply_discount(amount):                                     # VIOLATION 2: UDF for native math
    return amount * 0.9 if amount > 250 else amount

discounted = raw_orders.withColumn("net_amount", apply_discount(col("amount")))

discounted.cache()                                               # VIOLATION 3: cached, used ONCE
total = discounted.agg(F.sum("net_amount")).collect()[0][0]      # only downstream action

spark.conf.set("spark.sql.shuffle.partitions", "200")            # VIOLATION 4: default left
                                                                   # unexamined for this data size
by_customer = discounted.groupBy("customer_id").agg(F.sum("net_amount").alias("total"))
result_rows = by_customer.collect()                               # VIOLATION 5: collect() with no
                                                                   # bound -- fine at 100 rows here,
                                                                   # a landmine if customer_id
                                                                   # cardinality grows

print(f"total: {total:,.2f}, customers: {len(result_rows)}")

## Checklist pass 1 — is the optimizer blinded? (found via `explain()`)

In [ ]:
print("look for BatchEvalPython / un-starred UDF node:")
discounted.select("net_amount").explain(mode="formatted")
# VIOLATION 2 confirmed: native rewrite is `F.when(col("amount") > 250, col("amount") * 0.9)
#                                            .otherwise(col("amount"))`

## Checklist pass 2 — is caching justified? (12.0's three-condition rule)

In [ ]:
import urllib.request, json as _json
def storage_report():
    app_id = spark.sparkContext.applicationId
    url = f"http://localhost:4040/api/v1/applications/{app_id}/storage/rdd"
    rows = _json.load(urllib.request.urlopen(url))
    print(f"{len(rows)} cached object(s)" if rows else "nothing cached")
    for r in rows:
        print(" -", r["name"], "used by how many downstream actions? check the code, not the UI")

storage_report()
# VIOLATION 3 confirmed: `discounted` is cached but only ONE action (the groupBy) uses it
# after the sum() -- and sum() itself didn't even benefit since it ran BEFORE the cache
# materialized on first use. Delete the cache() call entirely.

## Checklist pass 3 — is skew present? (13.0's routine)

In [ ]:
discounted.groupBy("customer_id").count().collect()
url = f"http://localhost:4040/api/v1/applications/{spark.sparkContext.applicationId}/stages?status=complete"
last_stage = max(s["stageId"] for s in _json.load(urllib.request.urlopen(url)))
detail_url = f"http://localhost:4040/api/v1/applications/{spark.sparkContext.applicationId}/stages/{last_stage}"
tasks = list(_json.load(urllib.request.urlopen(detail_url))[0]["tasks"].values())
durs = sorted(t["duration"] for t in tasks)
n = len(durs)
print(f"median={durs[n//2]}ms max={durs[-1]}ms ratio={durs[-1]/max(durs[n//2],1):.1f}x")
# VIOLATION 1 confirmed: customer_id=1 holds 85% of rows -- classic hot key

## The fixed script — every violation addressed

Native expression instead of UDF; no wasted cache; salted skew fix
(10.4's technique, minimal form); shuffle partitions sized down for
this data volume; `collect()` guarded (13.1's pattern).

In [ ]:
def safe_collect(df, max_rows=100_000):
    n = df.count()
    if n > max_rows:
        raise ValueError(f"refusing to collect {n:,} rows")
    return df.collect()

spark.conf.set("spark.sql.shuffle.partitions", "16")             # FIX 4: sized for ~3M rows, not 200

fixed_orders = spark.range(3_000_000).withColumn(
    "customer_id", F.when(col("id") % 100 < 85, F.lit(1))
                     .otherwise((col("id") % 100).cast("int")))
fixed_orders = fixed_orders.withColumn("amount", (col("id") % 500).cast("double"))

fixed_discounted = fixed_orders.withColumn(                       # FIX 2: native, no UDF
    "net_amount", F.when(col("amount") > 250, col("amount") * 0.9).otherwise(col("amount")))

# FIX 1: salt the hot key (10.4's technique) before the skewed groupBy
salted = fixed_discounted.withColumn(
    "salt", F.when(col("customer_id") == 1, (F.rand() * 8).cast("int")).otherwise(F.lit(0)))
by_customer_salted = (salted.groupBy("customer_id", "salt")
                      .agg(F.sum("net_amount").alias("partial"))
                      .groupBy("customer_id")
                      .agg(F.sum("partial").alias("total")))

total_fixed = fixed_discounted.agg(F.sum("net_amount")).collect()[0][0]   # FIX 3: no cache, single use
result_rows_fixed = safe_collect(by_customer_salted)               # FIX 5: guarded

print(f"total: {total_fixed:,.2f}, customers: {len(result_rows_fixed)}")
print(f"matches original total? {abs(total_fixed - total) < 0.01}")

In [ ]:
spark.stop()

## Exercises

1. Before reading the "fixed script" cell, write down which of the
   five violations you spotted from the code alone, without running
   any diagnostics. Compare against what the UI evidence actually
   confirmed — did you find all five?
2. The salting fix adds a two-phase groupBy. Verify via `explain()`
   that the salted version's max-task-duration ratio (13.0) actually
   improves over the unsalted version for this specific 85%-hot-key
   dataset.
3. `spark.sql.shuffle.partitions=16` was chosen for "this data
   volume." Using 10.3's sizing math, justify (or challenge) that
   number for 3M rows.
4. Add a SIXTH deliberate violation to the bad script (pick one from
   13.3 or 13.4's territory — stale stats or wrong cluster shape) and
   write the checklist-pass code that would catch it.
5. Run the full checklist from 13.5's lesson against this fixed
   script. Is it actually ship-ready, or are there items the fix
   still doesn't address (e.g. error handling, config externalization
   from Module 17)? List what's still missing.

In [ ]:
# your exercise code here